# TERRA on the in-vivo perturb-FISH dataset -- sanity checks

Before running TERRA (`lotfollahi-lab/TERRA-96M`) on `pfish_terra.h5ad`, check:

1. **Encoder coverage**: how many of the 500 gene symbols in `pfish_terra.h5ad` map to Ensembl IDs
   that TERRA's pretrained tokenizer actually knows (`ensembl_dictionary.pkl` for symbol -> Ensembl ID,
   `token_dictionary.pkl` for the tokenizer's ~22k-gene vocab). This mirrors `scripts/terra/common.py`'s
   `token_delta()` (symbol -> Ensembl -> token lookup).

In [1]:
import pickle

import h5py
import numpy as np

PFISH_TERRA = "../../notebooks/in_vivo/pfish_terra.h5ad"   # 500 genes, full encoder-side panel
PFISH_FULL = "../../notebooks/in_vivo/pfish_full.h5ad"     # 154 genes, comparable to other models


def read_var_names(h5ad_path):
    # anndata.read_h5ad(..., backed="r") errors on this file's /uns/log1p (IORegistryError on
    # older anndata); var_names alone don't need anndata at all.
    with h5py.File(h5ad_path, "r") as f:
        idx = f["var"]["_index"][:]
    return [g.decode() if isinstance(g, bytes) else g for g in idx]


genes_500 = read_var_names(PFISH_TERRA)
genes_154 = read_var_names(PFISH_FULL)
print(f"pfish_terra.h5ad: {len(genes_500)} genes")
print(f"pfish_full.h5ad:  {len(genes_154)} genes")
assert set(genes_154) <= set(genes_500), "154-gene panel is not a subset of the 500-gene panel"
print("154-gene panel confirmed as a subset of the 500-gene panel.")

pfish_terra.h5ad: 500 genes
pfish_full.h5ad:  154 genes
154-gene panel confirmed as a subset of the 500-gene panel.


## 1. Encoder coverage of the 500 genes

Downloads the pretrained bundle (network access to the HF Hub required the first time; if
`lotfollahi-lab/TERRA-96M` is private you'll need `HF_TOKEN` set / `huggingface-cli login`).

In [2]:
from terra import download_pretrained

MODEL_REPO = "lotfollahi-lab/TERRA-96M"
LOCAL_DIR = "/data/a330d/projects/cellina-reproducibility/pretrained/"
model_dir = download_pretrained(MODEL_REPO, local_dir=LOCAL_DIR)
print("model bundle:", model_dir)

with open(f"{model_dir}/ensembl_dictionary.pkl", "rb") as f:
    ens_map = pickle.load(f)          # gene symbol -> Ensembl ID
with open(f"{model_dir}/token_dictionary.pkl", "rb") as f:
    token_dict = pickle.load(f)       # Ensembl ID (+ special tokens) -> token id

encoder_vocab = {k for k in token_dict if "ENS" in k}
print(f"tokenizer vocab: {len(encoder_vocab)} ENS-prefixed gene tokens (+ {len(token_dict) - len(encoder_vocab)} special tokens)")

/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import 

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/lotfollahi-lab/TERRA-96M/revision/main "HTTP/1.1 200 OK"


/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 887.87it/s]

model bundle: /data/a330d/projects/cellina-reproducibility/pretrained
tokenizer vocab: 21952 ENS-prefixed gene tokens (+ 1455 special tokens)


In [3]:
rows = []
for g in genes_500:
    ens_id = ens_map.get(g)
    rows.append((g, ens_id, ens_id in encoder_vocab if ens_id is not None else False))

no_ensembl = [g for g, e, ok in rows if e is None]
not_in_vocab = [g for g, e, ok in rows if e is not None and not ok]
covered = [g for g, e, ok in rows if ok]

print(f"{len(covered)}/{len(genes_500)} genes covered by TERRA's pretrained encoder vocab")
print(f"  {len(no_ensembl)} symbols have no Ensembl ID in ensembl_dictionary.pkl")
print(f"  {len(not_in_vocab)} symbols resolve to an Ensembl ID absent from token_dictionary.pkl")
if no_ensembl:
    print("no Ensembl mapping:", sorted(no_ensembl))
if not_in_vocab:
    print("Ensembl ID not tokenized:", sorted(not_in_vocab))

498/500 genes covered by TERRA's pretrained encoder vocab
  2 symbols have no Ensembl ID in ensembl_dictionary.pkl
  0 symbols resolve to an Ensembl ID absent from token_dictionary.pkl
no Ensembl mapping: ['TRAC', 'TRBC1']


## 2. LoRA fine-tune on `pfish_terra` (self-supervised, all cells)

Notebook version of `scripts/terra/finetuning_lora_pfish.py` -- same cells, same code, so you can
step through / inspect intermediate state instead of running it as a script. See that file's
docstring for the full rationale (px_to_um source, why this bypasses `common.load_dataset()`, the
`/uns/log1p` anndata workaround).

`EPOCHS`/`MAX_STEPS`/`BATCH_SIZE` below default to a **smoke test** (1 epoch, 5 steps, batch 32) --
bump them up once everything runs clean end to end and you've checked GPU memory headroom.

In [4]:
import json
import logging
import os
import re
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TQDM_DISABLE", "1")

import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import yaml
from datasets import load_from_disk

sys.path.insert(0, str(Path.cwd()))   # notebook cwd == scripts/terra, for `import common`
import common
import terra.training.finetune_self_supervised as fss
from terra.inference import embed_dataset, harmonize_adata
from terra.training.finetune_self_supervised import finetune_self_supervised, prepare_finetuned_model
from terra.utils.helper import init_model, parse_arch_kwargs, parse_protein_init_kwargs

# pfish_terra.h5ad has an /uns/log1p/base entry written with a "null" IOSpec encoding this anndata
# doesn't have a reader for -- harmless scanpy log1p-base marker, not read by anything below.
# Teach the registry to decode it as None instead of touching the file.
from anndata._io.specs.registry import _REGISTRY, IOSpec

if (h5py.Dataset, IOSpec("null", "0.1.0")) not in _REGISTRY.read:
    @_REGISTRY.register_read(h5py.Dataset, IOSpec("null", "0.1.0"))
    def _read_null(elem, _reader):
        return None

logging.basicConfig(level=logging.WARNING, format="%(message)s")
for name in ("terra.training.finetune_self_supervised", "terra.utils.helper"):
    logging.getLogger(name).setLevel(logging.INFO)

TARGETS = ["qkv", "proj", "fc1", "fc2"]   # LoRA-adapted submodules
EPS = 1e-6      # "changed" cannot be bitwise: EMA touches every tensor every step.

In [5]:
# --- config (smoke-test defaults; raise EPOCHS/MAX_STEPS/BATCH_SIZE once this runs clean) ---
ADATA_PATH = PFISH_TERRA
PX_TO_UM = 0.108     # pfish_eda (1).ipynb: um_per_pixel = 5.4 / 50
EPOCHS = 1
MAX_STEPS = 5        # None for a real run; caps cells to MAX_STEPS*BATCH_SIZE (1 epoch only)
BATCH_SIZE = 32
LR = 1e-4
SEED = 0
GUARD_CELLS = 20000
MAX_CELLS = None
NPROC = 16
FROM_RUN_DIR = None   # set to an existing run dir to skip training and only repack/embed/guard

SID = Path(ADATA_PATH).stem                                              # "pfish_terra"
WORK = Path("../../notebooks/in_vivo/terra_pfish/terra")                 # sibling to cellina_*_pfish dirs
RUN_DIR = WORK / "lora_run"
SELECTION = WORK / "epoch_selection.json"
TOK_CACHE = WORK.parent / "terra_tok"
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path(model_dir)
CFG = yaml.safe_load(open(MODEL_DIR / "model_config.yaml"))
TOKD = token_dict   # already loaded above from token_dictionary.pkl

In [6]:
def build_encoder():
    """The encoder exactly as embed.py builds it from the bundle's model_config.yaml."""
    n_special_tokens = len(CFG["meta"]["special_tokens"])
    seq_len = (CFG["data"]["seq_len_cell"] + CFG["data"]["seq_len_neighborhood"]
               + n_special_tokens)
    enc, _ = init_model(
        gt_type=CFG["meta"]["gt_type"], count_encoding=CFG["meta"]["count_encoding"],
        n_value_bins=CFG["meta"]["n_value_bins"], cell_pos_enc=CFG["meta"]["cell_pos_enc"],
        device="cpu", vocab_size=len(TOKD), seq_len=seq_len,
        n_special_tokens=n_special_tokens, n_segments=CFG["data"]["n_segments"],
        enc_emb_dim=CFG["meta"]["enc_emb_dim"], enc_depth=CFG["meta"]["enc_depth"],
        pred_emb_dim=CFG["meta"]["pred_emb_dim"], pred_depth=CFG["meta"]["pred_depth"],
        num_heads=CFG["meta"]["num_heads"], mlp_ratio=CFG["meta"]["mlp_ratio"],
        use_flash_attention=CFG["meta"]["use_flash_attention"],
        api_version=CFG["meta"]["api_version"],
        sep_gene_tokens_neb=CFG["data"]["sep_gene_tokens_neb"],
        predict_gene=CFG["meta"]["predict_gene"], pos_learnable=CFG["meta"]["pos_learnable"],
        n_special_values=CFG["data"].get("n_special_values", 0),
        nz_spc=CFG["data"].get("nz_spc", False),
        mlp_bias=CFG["meta"].get("mlp_bias", True),
        protein_init_kwargs=parse_protein_init_kwargs(CFG, TOKD),
        **parse_arch_kwargs(CFG))
    return enc

In [ ]:
def load_pfish_terra(adata_path, model_dir, px_to_um):
    """Mirror common.py's _terra_side() crc branch: pfish gene symbols are already human, so no
    merfish-style upper-case/ortholog step is needed -- just harmonize_adata's own symbol lookup."""
    raw = sc.read_h5ad(adata_path)
    raw.obs_names_make_unique()
    raw.obs["cell_id"] = raw.obs_names.astype(str)

    coords = np.asarray(raw.obsm["spatial"], dtype=np.float64) * px_to_um
    raw.obsm["spatial"] = coords
    ext = np.ptp(coords, axis=0)
    print(f"[terra] tissue extent: {ext[0]:.0f} x {ext[1]:.0f} um (px_to_um={px_to_um})")

    if sp.issparse(raw.X):
        raw.X = raw.X.tocsr()
    raw.layers["counts"] = raw.X.copy()
    raw = harmonize_adata(
        raw,
        gene_mapping_dict_file_path=f"{model_dir}/ensembl_dictionary.pkl",
        gene_occurrence_count_file_path=f"{model_dir}/gene_count_dictionary.pkl",
    )
    if sp.issparse(raw.X):
        raw.X = raw.X.tocsr()
    raw.layers["counts"] = raw.X.copy()     # re-sync after harmonize's gene/cell filtering
    return raw


adata_terra = load_pfish_terra(ADATA_PATH, str(MODEL_DIR), PX_TO_UM)
print(f"[harmonize] {adata_terra.n_obs:,} cells x {adata_terra.n_vars} genes survive TERRA harmonization")

In [8]:
tok_all = common.tokenize_cached(adata_terra, str(MODEL_DIR), TOK_CACHE, nproc=NPROC)
del adata_terra

INFO:terra.inference.tokenize:STEP 1: LOADING CONFIG...
INFO:terra.inference.tokenize:STEP 2: TOKENIZING ANNDATA OBJECT...
INFO:terra.tokenizers.cell_tokenizers:Loading token dictionary from /data/a330d/projects/cellina-reproducibility/pretrained/token_dictionary.pkl.
INFO:terra.tokenizers.cell_tokenizers:Filtering cells...
INFO:terra.tokenizers.cell_tokenizers:Computing spatial neighborhood...
INFO:terra.tokenizers.cell_tokenizers:Normalizing gene expression counts...
INFO:terra.tokenizers.cell_tokenizers:Retrieving gene tokens.
INFO:terra.tokenizers.cell_tokenizers:Ranking gene tokens based on normalized counts (sparse version).
INFO:terra.tokenizers.cell_tokenizers:Retrieving tokens for neighborhood cells.
INFO:terra.tokenizers.cell_tokenizers:Creating Hugging Face dataset...
INFO:terra.tokenizers.cell_tokenizers:Using generator for dataset creation.


OSError: Not enough disk space. Needed: Unknown size (download: Unknown size, generated: Unknown size

In [9]:
a = '/data/a330d/data/ood/trained/mintflow_joint/predictions_epoch_30.pkl'
mintflow_preds = pickle.load(open(a, "rb"))

In [15]:
mintflow_preds.keys()

dict_keys(['TissueSection 0 (zero-based)', 'TissueSection 1 (zero-based)', 'TissueSection 2 (zero-based)'])

In [16]:
mintflow_preds['TissueSection 2 (zero-based)']

{'MintFlow_Xint': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 2348080 stored elements and shape (111526, 2000)>,
 'MintFlow_Xmic': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 8640312 stored elements and shape (111526, 2000)>,
 'MintFlow_Xbar_int': array([[0.        , 0.        , 0.9381181 , ..., 3.5058415 , 1.5157166 ,
         0.1280453 ],
        [0.        , 0.        , 0.9213181 , ..., 3.5126467 , 1.5048054 ,
         0.10245441],
        [0.        , 0.        , 0.9054961 , ..., 3.5273182 , 1.5016879 ,
         0.10931811],
        ...,
        [0.        , 0.        , 0.9289654 , ..., 3.5074003 , 1.5031804 ,
         0.20372087],
        [0.        , 0.        , 0.9141639 , ..., 3.5151584 , 1.5012107 ,
         0.0852505 ],
        [0.        , 0.        , 0.9251101 , ..., 3.5163138 , 1.498574  ,
         0.14460927]], shape=(111526, 100), dtype=float32),
 'MintFlow_Xbar_mic': array([[0.8314887 , 0.6190072 , 0.429366  , ..., 1.2431068 , 0.   

In [ ]:
FT_TOK = TOK_CACHE
if MAX_STEPS:
    assert EPOCHS == 1, "MAX_STEPS implies a single epoch"
    MAX_CELLS = min(MAX_STEPS * BATCH_SIZE, MAX_CELLS or len(tok_all))
if MAX_CELLS and MAX_CELLS < len(tok_all):
    # ponytail: smoke only -- plain random subsample of rows; no cell-id filtering needed
    # because the self-supervised objective is label-free.
    FT_TOK = RUN_DIR / "tok_sub"
    if not FT_TOK.exists():
        rng = np.random.default_rng(SEED)
        idx = np.sort(rng.choice(len(tok_all), MAX_CELLS, replace=False))
        tok_all.select(idx).flatten_indices().save_to_disk(str(FT_TOK))
    tok_all = load_from_disk(str(FT_TOK))
N_CELLS = len(tok_all)
print(f"[tok] {N_CELLS} rows from {FT_TOK} | columns {list(tok_all.features)}", flush=True)

In [ ]:
# fss._build_model calls init_model without the config-dependent extras (nz_spc / n_special_values /
# protein_init / arch kwargs), same gap finetuning.py patches.
_orig_init_model = fss.init_model


def _patched_init_model(**kw):
    kw["nz_spc"] = CFG["data"].get("nz_spc", False)
    kw["mlp_bias"] = CFG["meta"].get("mlp_bias", True)
    kw["n_special_values"] = CFG["data"].get("n_special_values", 0)
    kw["protein_init_kwargs"] = parse_protein_init_kwargs(CFG, TOKD)
    kw.update(parse_arch_kwargs(CFG))
    return _orig_init_model(**kw)


fss.init_model = _patched_init_model

# fss.init_cell_dataset also omits `pad_special_tokens=True`, which embed_dataset always sets:
# with special tokens (112M: ["batch"]) the loader would otherwise look up a per-cell
# `batch_value` -- a corpus batch identity (`spv_{dataset_id}_{batch}`) that new slides cannot
# have.  Padding the slot is exactly what inference does, so train and embed see the same input.
_orig_init_cell_dataset = fss.init_cell_dataset


def _patched_init_cell_dataset(**kw):
    kw.setdefault("pad_special_tokens", True)
    kw.setdefault("truncate_neighbors", CFG["data"].get("truncate_neighbors", False))
    kw.setdefault("tokenized_seq_len_cell", CFG["data"].get("tokenized_seq_len_cell", None))
    return _orig_init_cell_dataset(**kw)


fss.init_cell_dataset = _patched_init_cell_dataset

In [ ]:
# preflight: the pretrained target-encoder weights must load into build_encoder() with no key
# mismatch, before we spend time fine-tuning.
PRETRAINED = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["target_encoder"]
PRETRAINED = {k.replace("module.", ""): v for k, v in PRETRAINED.items()}
_enc = build_encoder()
_enc.load_state_dict(PRETRAINED)     # strict; raises on any mismatch
n_total = sum(p.numel() for p in _enc.parameters())
print(f"[preflight] ALL KEYS MATCHED | {n_total:,} encoder params", flush=True)
del _enc

In [ ]:
# log capture + training clock
epoch_log = []
_EPOCH_RE = re.compile(r"Epoch \[(\d+)/\d+\], Loss: ([\d.]+), LR: ([\d.e+-]+)")


class _EpochLog(logging.Handler):
    def emit(self, record):
        m = record.getMessage()
        g = _EPOCH_RE.search(m)
        if g:
            epoch_log.append({"line": m, "t": time.time(), "epoch": int(g[1]),
                              "loss": float(g[2]), "lr": float(g[3])})


logging.getLogger("terra.training.finetune_self_supervised").addHandler(_EpochLog())

# Start the clock at the first real step so encoder build / dataloader spin-up is
# excluded from sec_per_step. apply_masks runs once per step inside the loop.
first_step = []
_orig_apply_masks = fss.apply_masks


def _timed_apply_masks(*a, **kw):
    if not first_step:
        first_step.append(time.time())
    return _orig_apply_masks(*a, **kw)


fss.apply_masks = _timed_apply_masks

In [ ]:
# fine-tune -- Xenium tutorial recipe except lr (see LR) and warmup (0.1 epoch: our epochs are
# ~50x longer than the tutorial's, so a 1-epoch warmup never reaches peak)
finetune_args = {
    "model": {"pretrained_checkpoint_path": str(MODEL_DIR),
              "finetune_checkpoint_path": str(RUN_DIR)},
    "data": {"finetune_dataset": [str(FT_TOK)], "batch_size": BATCH_SIZE,
             "num_workers": 8, "pin_memory": True, "drop_last": True,
             "sample_segments": False, "sample_gene_masks": True},
    "finetune": {"num_epochs": EPOCHS, "lr": LR, "start_lr": LR / 10, "final_lr": LR / 10,
                 "warmup_epochs": 0.1, "weight_decay": 0.04, "final_weight_decay": 0.4,
                 "ema_momentum": 0.9995, "final_ema_momentum": 1.0,
                 "loss_fn_type": "smooth_l1", "clip_grad": 2.0, "use_bfloat16": True,
                 "use_peft": True, "peft_method": "lora", "peft_rank": 16,
                 "peft_alpha": 256, "peft_dropout": 0.1, "peft_bias": "none",
                 "peft_target_modules": TARGETS, "save_every": 1},
}
steps_per_epoch = N_CELLS // BATCH_SIZE          # drop_last=True

if FROM_RUN_DIR:
    run_dir = Path(FROM_RUN_DIR)
    wall = peak_gib = float("nan")
    old = next(iter(sorted(run_dir.parents[1].glob("*bundle*/ft_summary.json"))), None)
    prev = json.loads(old.read_text()) if old else {}
    loss_per_epoch = prev.get("loss_per_epoch", [])
    steps_per_epoch = prev.get("steps_per_epoch", steps_per_epoch)
    print(f"[reuse] {run_dir} | prior summary: {old}", flush=True)
else:
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    run_dir = Path(finetune_self_supervised(args=finetune_args, save_folder_path=str(RUN_DIR),
                                            run_name="run"))
    wall = time.time() - t0
    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    assert len(epoch_log) == EPOCHS, f"{len(epoch_log)} epoch lines for {EPOCHS} epochs"
    loss_per_epoch = [e["loss"] for e in epoch_log]
    print(f"[train] {[e['line'] for e in epoch_log]}", flush=True)
    print(f"[probe] peak_gpu_gib={peak_gib:.1f} sec_per_step={(wall - (first_step[0] - t0)) / max(1, steps_per_epoch * EPOCHS):.2f}",
          flush=True)

In [ ]:
ckpts = {int(f.stem.split("_")[-1]): f for f in run_dir.glob("checkpoint_epoch_*.pt")}
assert ckpts, f"no checkpoint_epoch_*.pt in {run_dir}"
FINAL_EPOCH = max(ckpts)
print(f"[epochs] checkpoints {sorted(ckpts)} in {run_dir}", flush=True)

(RUN_DIR / "ft_config.json").write_text(json.dumps(
    {"sid": SID, "n_cells": N_CELLS, "epochs": EPOCHS, "max_steps": MAX_STEPS, "batch_size": BATCH_SIZE,
     "lr": LR, "steps_per_epoch": steps_per_epoch, "finetune_dataset": str(FT_TOK),
     "run_dir": str(run_dir), "terra_args": finetune_args}, indent=2, default=str))

### Per-epoch: repack -> verify -> embed every cell -> collapse-guard stats

In [ ]:
PRE_ONLINE = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["encoder"]
PRE_ONLINE = {k.replace("module.", ""): v for k, v in PRE_ONLINE.items()}


def repack(epoch, bundle):
    """prepare_finetuned_model for one epoch + the structural checks (these still assert:
    they are correctness, not collapse)."""
    if bundle.exists():
        shutil.rmtree(bundle)
    prepare_finetuned_model(finetuned_checkpoint_dir=str(run_dir), pretrained_model_dir=str(MODEL_DIR),
                            output_dir=str(bundle), checkpoint_epoch=epoch, use_peft=True)
    for f in ("model_config.yaml", "token_dictionary.pkl", "ensembl_dictionary.pkl",
              "gene_count_dictionary.pkl", "model_checkpoint.pt"):
        assert (bundle / f).exists(), f"bundle is missing {f}"
    new_sd = torch.load(bundle / "model_checkpoint.pt", map_location="cpu")["target_encoder"]
    new_sd = {k.replace("module.", ""): v for k, v in new_sd.items()}
    build_encoder().load_state_dict(new_sd)     # strict; raises on any mismatch
    assert set(new_sd) == set(PRETRAINED), "repacked key set differs from pretrained"

    deltas = {k: (new_sd[k].float() - PRETRAINED[k].float()).abs().max().item()
              for k in PRETRAINED if PRETRAINED[k].is_floating_point()}
    changed = sorted(k for k, v in deltas.items() if v > EPS)
    lora_changed = [k for k in changed if any(t in k for t in TARGETS)]
    off_target = [k for k in changed if k not in lora_changed]
    assert lora_changed, "no LoRA-target tensor changed -- fine-tuning had no effect"
    per_block = {b: sum(f".blocks.{b}." in k for k in lora_changed)
                 for b in range(CFG["meta"]["enc_depth"])}
    assert not [b for b, n in per_block.items() if n == 0], f"empty LoRA blocks: {per_block}"
    # Off-target tensors move too, and it is NOT round-off: the exported target encoder is an
    # EMA of the ONLINE encoder, which starts from checkpoint['encoder'] -- a different tensor
    # set from checkpoint['target_encoder'] in the pretrained bundle.  So every frozen weight
    # drifts along that pretrained online/EMA gap and can move no further than it.
    gap = {k: (PRE_ONLINE[k].float() - PRETRAINED[k].float()).abs().max().item() for k in deltas}
    floor = {k: 1e-3 * PRETRAINED[k].float().abs().max().item() for k in deltas}
    unexplained = [k for k in off_target if deltas[k] > 1.01 * gap[k] + max(EPS, floor[k])]
    assert not unexplained, f"off-target tensors moved beyond the pretrained EMA gap: {unexplained[:5]}"
    drift = max((deltas[k] / max(gap[k], 1e-12) for k in off_target), default=0.0)
    print(f"[ep{epoch}] repack {bundle} | {len(changed)}/{len(deltas)} tensors changed: "
          f"{len(lora_changed)} LoRA targets, per-block {per_block}; {len(off_target)} frozen "
          f"tensors EMA-drifted at most {drift:.1%} of the pretrained encoder/target gap", flush=True)
    return {"n_changed_tensors": len(changed), "n_lora_changed": len(lora_changed),
            "lora_changed_per_block": per_block, "n_off_target_drifted": len(off_target),
            "max_off_target_drift_frac_of_ema_gap": drift}


def rank_std(x):
    """(effective rank = exp entropy of the PCA spectrum, mean per-dim std, mean cell-cell cosine)"""
    xc = x - x.mean(0)
    s_ = np.linalg.svd(xc, compute_uv=False) ** 2
    p_ = s_ / s_.sum()
    eff_rank = float(np.exp(-(p_ * np.log(p_ + 1e-12)).sum()))
    xn = x / np.linalg.norm(x, axis=1, keepdims=True)
    cc = xn @ xn.T
    return eff_rank, float(xc.std(0).mean()), float(cc[np.triu_indices(len(x), 1)].mean())

In [ ]:
# The guard compares the SAME fixed subsample of cells frozen vs fine-tuned; frozen is embedded once.
rng_ = np.random.default_rng(SEED)
GUARD_ROWS = np.sort(rng_.choice(N_CELLS, min(GUARD_CELLS, N_CELLS), replace=False))
tok_guard = tok_all.select(GUARD_ROWS)
frozen = embed_dataset(dataset=tok_guard, model_folder_path=str(MODEL_DIR),
                       **dict(common.EMB_KWARGS, num_workers=4))
frozen = {k: np.asarray(frozen[k], dtype=np.float64) for k in common.EMB_KEYS}

CRITERIA = {"eff_rank": "> 0.5x frozen", "mean_dim_std": "> 0.5x frozen",
            "mean_cosine_to_frozen": "> 0.5"}
epochs = {}
for ep in sorted(ckpts):
    ep_dir = WORK / f"lora_ep{ep}"
    stats = {"repack": repack(ep, ep_dir / "lora_bundle")}
    emb = embed_dataset(dataset=tok_guard, model_folder_path=str(ep_dir / "lora_bundle"),
                        **dict(common.EMB_KWARGS, num_workers=4))
    for k in common.EMB_KEYS:
        a_, b_ = frozen[k], np.asarray(emb[k], dtype=np.float64)
        ok = np.isfinite(b_).all()
        cos = float(np.mean((a_ * b_).sum(1) / (np.linalg.norm(a_, axis=1) * np.linalg.norm(b_, axis=1))))
        mx = float(np.abs(a_ - b_).max())
        (r0, s0, c0), (r1, s1, c1) = rank_std(a_), rank_std(b_)
        stats[k] = {"finite": bool(ok), "max_abs_delta": mx, "mean_cosine_to_frozen": cos,
                    "eff_rank": [r0, r1], "mean_dim_std": [s0, s1],
                    "mean_cell_cell_cosine": [c0, c1],
                    "pass": {"eff_rank": bool(r1 > 0.5 * r0), "mean_dim_std": bool(s1 > 0.5 * s0),
                             "mean_cosine_to_frozen": bool(cos > 0.5)}}
        print(f"[ep{ep}] {k}: max|d| {mx:.4e} | cos-to-frozen {cos:.4f} | eff rank {r0:.1f}->{r1:.1f} "
              f"| dim std {s0:.4f}->{s1:.4f} | cell-cell cos {c0:.3f}->{c1:.3f}", flush=True)
    stats["passed"] = bool(all(v for k in common.EMB_KEYS for v in stats[k]["pass"].values())
                           and all(stats[k]["finite"] for k in common.EMB_KEYS))
    print(f"[ep{ep}] guard passed={stats['passed']}", flush=True)
    epochs[str(ep)] = stats
    del emb

passing = [e for e in sorted(ckpts) if epochs[str(e)]["passed"]]

In [ ]:
# epoch_selection.json -- mirrors the CRC queue's format
SELECTION.write_text(json.dumps({
    "sid": SID, "adata_path": ADATA_PATH, "run_dir": str(run_dir),
    "final_epoch": FINAL_EPOCH, "latest_passing_epoch": (max(passing) if passing else None),
    "criteria": CRITERIA, "guard_cells": int(len(GUARD_ROWS)), "n_cells": N_CELLS, "steps_per_epoch": steps_per_epoch,
    "loss_per_epoch": loss_per_epoch, "wall_seconds": wall, "peak_gpu_gib": peak_gib,
    "lr": LR, "batch_size": BATCH_SIZE, "epochs": epochs,
    "model_repo": MODEL_REPO, "model_dir": str(MODEL_DIR), "px_to_um": PX_TO_UM,
}, indent=2))
print(json.dumps({"final_epoch": FINAL_EPOCH,
                  "latest_passing_epoch": (max(passing) if passing else None),
                  "passed": {e: epochs[e]["passed"] for e in epochs},
                  "loss_per_epoch": loss_per_epoch}, indent=2), flush=True)
print("wrote", SELECTION, flush=True)